# Contradiction-feedback debate — pilot
Select a GPU runtime (Runtime > Change runtime type). Run cells in order, and upload the prototype ZIP when asked. The optional smoke test checks the first four questions. The pilot uses the first 25 shared question IDs. Results are saved to your Drive.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os
import zipfile
import io
drive.mount('/content/drive')
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one prototype ZIP.'
required = ('runner.py', 'question_ids.json', 'requirements.txt', 'test_runner.py', 'contradiction_debate.py', 'test_contradiction_debate.py')
work = Path('/content/contradiction_pilot')
work.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for name in required:
        (work / name).write_bytes(archive.read(name))
os.chdir(work)
%pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
print('GPU:', torch.cuda.get_device_name(0))
!python -m unittest -v

In [ ]:
# Optional: run this cell to smoke-test four questions.
output = '/content/drive/MyDrive/TeamError404/contradiction-smoke-v1'
!python -u contradiction_debate.py --limit 4 --question-batch-size 4 --output "{output}"

In [ ]:
# Main pilot. Run again with the same directory to resume.
output = '/content/drive/MyDrive/TeamError404/contradiction-pilot-v1'
!python -u contradiction_debate.py --limit 25 --question-batch-size 4 --output "{output}"

In [ ]:
import json
summary = Path('/content/drive/MyDrive/TeamError404/contradiction-pilot-v1/summary.json')
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))
    files.download(str(summary))
else:
    print('Pilot not completed yet; resume the previous cell.')